### AI Correction

The confidence from Textract gives us an opportunity to identify low confidence OCR. One way to increase the likelihood of correct answer is to pass the low confidence through another AI model such as a vision language model (VLM). You may choose any VLM for this step - for instance a new VLM focused on OCR has been recently released by Nanonets. In this tutorial we will use Google's Gemini to demonstrate.

<p align="center">
  <img 
    src="assets/ai_correction.png" 
    alt="Logo" 
    width="400" 
    style="height: auto;" 
  />
</p>

In [1]:
import uuid
from pydantic import BaseModel, Field
from langchain.schema import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import dotenv
import os
from trp import trp2
import json

dotenv.load_dotenv(override=True)


True

#### Low confidence OCR

In [2]:

path_to_pdf = "data/processed_pdf/WO2020245233A1.pdf"
path_to_textract_json = "data/raw_textract_json/WO2020245233A1.json"


In [3]:


from ai_correction_utils import get_document_text_image_pairs


class CellTextImagePairs(BaseModel):
    id: str = Field(description="The ID of the cell", default_factory=lambda: str(uuid.uuid4())[:4])
    cell_id: str = Field(description="The ID of the cell in the document")
    text: str = Field(description="The text content of the cell")
    image: str = Field(description="The base64 encoded image of the cell")
    confidence: float = Field(description="The confidence score of the OCR text extraction", default=100.0)

# Load the textract JSON
textract_json = json.loads(open(path_to_textract_json).read())
pdf_bytes = open(path_to_pdf, mode="rb").read()
doc = trp2.TDocumentSchema().load(textract_json) 

# Filter out low confidence cells
ocr_image_pairs: list[CellTextImagePairs] = get_document_text_image_pairs(doc, pdf_bytes, confidence_threshold=80)



In [ ]:
# Needed for later - mapping cell IDs to short IDs after correction
cell_id_short_id_map = {pair.id: pair.cell_id for pair in ocr_image_pairs}

In [ ]:
from ai_correction_utils import display_ocr_image_pair

display_ocr_image_pair(ocr_image_pairs)

#### Prompt

In [ ]:
INTRO = """
1. You are provided with a list of images - low confidence OCR.
2. You are an OCR-like text extraction tool which corrects the OCR-extracted text from provided images.
3. Please format the text in LaTeX if required to express superscripts, subscripts and symbols.
4. The cell provided may be empty, in which case you should return an empty string.
. Image may contain nucleotide sequences (Letters of A, T, C, G, U) or variations of nucleotides with chemical \
modifications.
. Be aware of repeating letters as this is often seen in oligonucleotide sequences.
7. Do not hallucinate or interpolate data.

"""

intro_prompt = [("system", INTRO)]


#### LLM Model

In [ ]:

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


#### Multiple images per prompt

In [ ]:
class CorrectedResponseId(BaseModel):
    """Corrected response"""

    id: str = Field(description="The Query ID of the image.")
    text: str = Field(description="The corrected text")

class Response(BaseModel):
    OCR: list[CorrectedResponseId] = Field(
        description="The list of corrected responses for each image.",
    )
    def __iter__(self):
        return iter(self.OCR)
    

In [ ]:
from ai_correction_utils import batched

output_model = llm.with_structured_output(Response, method="function_calling")
results_list = []

for batch_images in batched(ocr_image_pairs, 5):
    messages = [SystemMessage(content="QUERIES: \nPlease provide the corrected OCR for the following images:\n")]
    
    for img in batch_images:
        message = HumanMessage(
            content=[
                {"type": "text", "text": "Image ID: " + img.id},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img.image}"}},
            ]
        )  

        messages.append(message)    
    try:
        response: Response = output_model.invoke(intro_prompt + messages)
    except Exception:
        break
    
    if not response:
        continue

    results_list += response.OCR


id_correction_map = {}
for corrected_response in results_list:
    if not isinstance(corrected_response, Exception):
        cell_id = cell_id_short_id_map[corrected_response.id]
        id_correction_map[cell_id] = corrected_response



In [ ]:
import ai_correction_utils
ai_correction_utils.visualize_differences_with_image(
    {im.cell_id: im for im in ocr_image_pairs},
    id_correction_map,
)

In [ ]:
from ai_correction_utils import merge_corrections

doc = merge_corrections(doc, id_correction_map,)

In [ ]:
# Save the corrected document

corrected_textract_json = trp2.TDocumentSchema().dump(doc)
with open("data/corrected_textract_json/WO2020245233A1.json", "w") as f:
    json.dump(corrected_textract_json, f, indent=2)
